# TabPFN-3 on Amazon SageMaker — sample notebook

End-to-end walkthrough for buyers of the **TabPFN-3** or **TabPFN-3 (+Thinking)** AWS Marketplace listings.
Covers: subscription → deploy as a SageMaker async-inference endpoint → invoke via the `tabpfn_client.sagemaker` Python SDK → clean up.

**Prerequisites**

- An active AWS Marketplace subscription to one of the listings: **TabPFN-3** (base SKU — single forward pass) or **TabPFN-3 (+Thinking)** (full SKU — base + optional Thinking mode).
- An AWS account with permission to create SageMaker `Model`, `EndpointConfig`, and `Endpoint` resources, and to read/write the S3 bucket you use for async input/output.
- A SageMaker execution role ARN that trusts `sagemaker.amazonaws.com` and has read access to that S3 bucket.
- `pip install --upgrade 'tabpfn-client[sagemaker]'`

The notebook is identical for both SKUs; the only difference is which Model Package ARN you paste below and whether you pass `thinking_effort` at invoke time.

## 1. Configuration

After subscribing to the listing, AWS shows you the Model Package ARN in the *Continue to configuration* step.
Copy the ARN for your AWS region and paste it below. The ARN looks like
`arn:aws:sagemaker:us-east-1:<acct>:model-package/<some-uuid>`.

In [ ]:
import os
import time

import boto3

# >>> EDIT THESE <<<
MODEL_PACKAGE_ARN = "arn:aws:sagemaker:us-east-1:<account>:model-package/<id>"
EXECUTION_ROLE_ARN = "arn:aws:iam::<account>:role/<SageMakerExecutionRole>"
ASYNC_S3_BUCKET = "<your-async-io-bucket>"  # bucket the execution role can read+write
REGION = "us-east-1"

# Derived names (no need to change)
MODEL_NAME = "tabpfn-3-sample"
ENDPOINT_CONFIG_NAME = f"{MODEL_NAME}-cfg"
ENDPOINT_NAME = f"{MODEL_NAME}-endpoint"

sm = boto3.client("sagemaker", region_name=REGION)

## 2. Deploy as a SageMaker Async Inference endpoint

Async inference is the recommended deployment for any non-trivial dataset.
Synchronous real-time invocations cap payloads at 6 MB and processing at 60 seconds,
which tabular workloads regularly exceed. Async lifts both caps (1 GB payload,
60 min processing) at the cost of an S3-mediated request/response round-trip.

In [ ]:
sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=EXECUTION_ROLE_ARN,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
)

sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.g5.xlarge",
            "InitialVariantWeight": 1.0,
            "ModelDataDownloadTimeoutInSeconds": 600,
            "ContainerStartupHealthCheckTimeoutInSeconds": 600,
        }
    ],
    AsyncInferenceConfig={
        "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 4},
        "OutputConfig": {
            "S3OutputPath": f"s3://{ASYNC_S3_BUCKET}/tabpfn-3-async/outputs/",
            "S3FailurePath": f"s3://{ASYNC_S3_BUCKET}/tabpfn-3-async/failures/",
        },
    },
)

sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME)

print("Endpoint creating; waiting for InService (typically 6-10 minutes)...")
sm.get_waiter("endpoint_in_service").wait(EndpointName=ENDPOINT_NAME)
print("Endpoint is InService.")

## 3. Invoke the endpoint with the `tabpfn_client` Python SDK

The `tabpfn_client.sagemaker.TabPFNClassifier` / `TabPFNRegressor` classes wrap
`boto3.client("sagemaker-runtime")` with a scikit-learn-style API and handle the
async staging-input / poll-output round-trip for you.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from tabpfn_client.sagemaker import TabPFNClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = TabPFNClassifier(
    endpoint_name=ENDPOINT_NAME,
    region_name=REGION,
    use_async=True,
    s3_bucket=ASYNC_S3_BUCKET,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)
print("prediction shape:", proba.shape, " accuracy:", (y_pred == y_test).mean())

## 4. Thinking mode (TabPFN-3 (+Thinking) SKU only)

If you subscribed to the **TabPFN-3 (+Thinking)** listing, the same endpoint also accepts
an optional `thinking_effort` parameter (`"medium"` or `"high"`) that engages Thinking mode
for that request. Thinking mode applies additional inference-time computation on top of
TabPFN-3 to push prediction quality further. Omit `thinking_effort` to use the base
single-forward-pass behavior — both modes work on the same endpoint and are selected per call.

Skip this cell if you subscribed to the base TabPFN-3 listing (the server returns 422 on
thinking requests for the base SKU).

In [ ]:
clf_thinking = TabPFNClassifier(
    endpoint_name=ENDPOINT_NAME,
    region_name=REGION,
    use_async=True,
    s3_bucket=ASYNC_S3_BUCKET,
    thinking_effort="medium",  # or "high"
)
clf_thinking.fit(X_train, y_train)

y_pred_t = clf_thinking.predict(X_test)
proba_t = clf_thinking.predict_proba(X_test)
print("thinking prediction shape:", proba_t.shape, " accuracy:", (y_pred_t == y_test).mean())

## 5. Clean up

Endpoints bill per instance-hour while they're `InService`. Tear down once you're done.

In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
sm.delete_model(ModelName=MODEL_NAME)
print("Endpoint + config + model deleted.")